# 00 setup

Run once per Colab session. Runtime -> Change runtime type -> T4 GPU.

Everything on Drive lives under `MyDrive/Colab Notebooks/tfm/` (`ROOT` below). This notebook sits in `ROOT/notebooks/` and is opened from Drive with Colab. Code is pulled from GitHub with a single-repo fine-grained token (Contents: read and write, 30-day expiry) stored in Colab Secrets as `GH_TOKEN`. The push cell copies small results and the notebooks (outputs cleared) back into the repo. Checkpoints and data stay on Drive.

In [ ]:
import os, sys, glob, shutil
try:
    from google.colab import drive, userdata                      # Colab: mount Drive, pull the repo, sync new notebooks to Drive
    drive.mount("/content/drive")
    tok = userdata.get("GH_TOKEN")
    if not os.path.exists("/content/transaction-fm"):
        !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
    else:
        !cd /content/transaction-fm && git pull -q
    %pip install -q datasets xgboost
    %cd /content/transaction-fm
except ImportError:                                               # laptop: this notebook is the repo copy; data under $TFM_ROOT
    here = os.getcwd()
    while not os.path.isdir(os.path.join(here, "src", "tfm")):
        assert here != os.path.dirname(here), "run this notebook from inside the transaction-fm repo"
        here = os.path.dirname(here)
    %cd {here}
sys.path.insert(0, os.path.abspath("src"))
from tfm.runtime import root, describe, on_colab
ROOT = root()
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
if on_colab():
    for f in glob.glob("notebooks/*.ipynb"):                      # new notebooks from the repo appear in Drive
        dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
        if not os.path.exists(dst):
            shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok |", describe(), "| ROOT =", ROOT)

## Check the GPU and the Drive layout

In [ ]:
import os, torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
for d in ["notebooks", "data/raw", "data/processed", "pretrain", "finetune", "xgb", "results"]:
    p = f"{ROOT}/{d}"
    os.makedirs(p, exist_ok=True)
    print(f"{d:16s}", sorted(os.listdir(p))[:10])

`data/raw` should list the two Sparkov csv files (`fraudTrain.csv`, `fraudTest.csv`) before step 1.

## Push results and notebooks back to GitHub

Run after a step has written json or png into `ROOT/results/`. Copies the small result files and every notebook in `ROOT/notebooks/` (outputs kept, so GitHub renders them) into the repo, then pushes. Checkpoints never leave Drive.

In [ ]:
import shutil, glob, os, json
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
if on_colab():
    def same_code(a, b):   # a Drive notebook may be copied back only if its code cells equal the repo version
        cells = lambda p: ["".join(c["source"]) for c in json.load(open(p))["cells"] if c["cell_type"] == "code"]
        return cells(a) == cells(b)
    for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
        repo = f"notebooks/{os.path.basename(f)}"
        if not os.path.exists(repo) or same_code(f, repo):
            shutil.copy(f, repo)
        else:
            print("STALE, not copied:", os.path.basename(f), "-> delete it from Drive and rerun the setup cell")
    !git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
    !git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
    !git log --oneline | head -1
else:
    print("laptop: results copied into results/; this notebook file already is the repo copy; commit from the terminal")
    !git status --short results notebooks

## End of session

In [ ]:
if on_colab():
    drive.flush_and_unmount()